# AST Laughter Labeler for 620 Videos

Uses **MIT/ast-finetuned-audioset-10-10-0** (1.2M downloads) to detect laughter in 10-second audio clips.

AudioSet includes 'Laughter' class! This gives us proper labels for our 620 videos.

In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive')

!pip install -q torch torchaudio librosa numpy pandas
!pip install -q transformers accelerate

import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import json
import os
import librosa

BASE = '/content/drive/MyDrive/chuckle_net'
AUDIO_DIR = f'{BASE}/audio'
OUTPUT_DIR = f'{BASE}/ast_labels'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Audio dir: {AUDIO_DIR}')
print(f'Output dir: {OUTPUT_DIR}')

# List audio files
audio_files = [f for f in os.listdir(AUDIO_DIR) if f.endswith('.m4a')]
print(f'Found {len(audio_files)} audio files')

In [ ]:
# === LOAD AST MODEL ===
# Using Audio Spectrogram Transformer fine-tuned on AudioSet
# AudioSet has 527 classes including 'Laughter'

print('Loading AST model...')
model_name = 'MIT/ast-finetuned-audioset-10-10-0.4593'

from transformers import AutoModelForAudioClassification, AutoFeatureExtractor

# IMPORTANT: AST uses feature extractor to convert audio to log-mel spectrogram
feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)
model = AutoModelForAudioClassification.from_pretrained(model_name)
model.eval()

# Find laughter class in AudioSet
laughter_class = None
if hasattr(model.config, 'id2label'):
    print(f'Model has {len(model.config.id2label)} classes')
    for cid, label in model.config.id2label.items():
        if 'laugh' in label.lower():
            laughter_class = int(cid)
            print(f'Found laughter class: {cid} = {label}')
            break

if laughter_class is None:
    # AudioSet class 261 is typically Laughter
    laughter_class = 261
    print(f'Using fallback laughter class: 261')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f'Model loaded on {device}')

In [ ]:
# === PROCESS AUDIO FILE (FASTER - LOAD ONCE, SLICE) ===
def process_video_fast(audio_path, video_id, clip_duration=10.0, overlap=2.0):
    """Load audio ONCE, extract clips, predict, return results."""
    try:
        # Load entire audio file ONCE
        y, sr = librosa.load(audio_path, sr=16000, mono=True)
        duration = len(y) / sr
        
        # Extract clip start/end times
        clip_times = []
        for start in np.arange(0, duration - clip_duration, overlap):
            clip_times.append((start, start + clip_duration))
        
        if not clip_times:
            return None
        
        # Prepare batch of waveforms
        waveforms = []
        valid_times = []
        
        for start, end in clip_times:
            start_sample = int(start * sr)
            end_sample = int(end * sr)
            clip_y = y[start_sample:end_sample]
            
            # Pad if short
            if len(clip_y) < int(clip_duration * sr):
                clip_y = np.pad(clip_y, (0, int(clip_duration * sr) - len(clip_y)))
            
            waveforms.append(clip_y)
            valid_times.append((start, end))
        
        # Stack into batch: (batch, samples)
        batch_waveform = np.stack(waveforms, axis=0)  # (N, 160000)
        
        # Use feature extractor to convert to log-mel spectrogram
        # This is what AST needs!
        inputs = feature_extractor(
            batch_waveform, 
            sampling_rate=16000, 
            return_tensors='pt'
        )
        
        # Move to device
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Predict
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits  # (batch, num_classes)
            probs = torch.softmax(logits, dim=-1)
            laughter_probs = probs[:, laughter_class].cpu().numpy()
        
        # Build result
        result = {
            'video_id': video_id,
            'n_clips': len(valid_times),
            'clips': [
                {'start': t[0], 'end': t[1], 'laughter_prob': float(laughter_probs[i])}
                for i, t in enumerate(valid_times)
            ],
            'mean_laugh_prob': float(np.mean(laughter_probs)),
            'max_laugh_prob': float(np.max(laughter_probs)),
            'n_positive': int((laughter_probs > 0.1).sum())
        }
        
        return result
        
    except Exception as e:
        print(f'Error processing {video_id}: {e}')
        return None

print('process_video_fast() defined')

In [ ]:
# === PROCESS ALL VIDEOS ===
# This is the main loop - runs on CPU
# Expected time: ~3-5 min per video on CPU = ~30-50 hours total
# For speed, we recommend running on Colab GPU

all_results = []

for audio_file in tqdm(audio_files, desc='Processing videos'):
    video_id = audio_file.replace('.m4a', '')
    output_file = f'{OUTPUT_DIR}/{video_id}.json'
    
    # Skip if already processed
    if os.path.exists(output_file):
        with open(output_file) as f:
            all_results.append(json.load(f))
        continue
    
    audio_path = f'{AUDIO_DIR}/{audio_file}'
    
    # Process video
    result = process_video_fast(audio_path, video_id)
    
    if result:
        # Save result
        with open(output_file, 'w') as f:
            json.dump(result, f)
        all_results.append(result)
    
    # Progress update every 10 videos
    if len(all_results) % 10 == 0:
        n_done = len(all_results)
        avg_pos = np.mean([r['n_positive'] for r in all_results])
        print(f'\nProgress: {n_done}/{len(audio_files)} videos, avg positive clips: {avg_pos:.1f}')

print(f'\nProcessed {len(all_results)} videos')

In [ ]:
# === ANALYZE RESULTS ===
df = pd.DataFrame(all_results)
print('=== LABEL STATISTICS ===')
print(f'Total videos: {len(df)}')
print(f'Mean laughter prob: {df["mean_laugh_prob"].mean():.4f}')
print(f'Max laughter prob: {df["max_laugh_prob"].max():.4f}')
print(f'Videos with high prob (>0.1): {(df["max_laugh_prob"] > 0.1).sum()}')
print(f'Videos with high prob (>0.5): {(df["max_laugh_prob"] > 0.5).sum()}')

# Overall clip-level stats
all_clips = []
for r in all_results:
    for clip in r['clips']:
        all_clips.append({
            'video_id': r['video_id'],
            'start': clip['start'],
            'end': clip['end'],
            'laughter_prob': clip['laughter_prob']
        })
clips_df = pd.DataFrame(all_clips)
print(f'\nTotal clips: {len(clips_df)}')
print(f'Clips with laughter (>0.1): {(clips_df["laughter_prob"] > 0.1).sum()}')
print(f'Positive rate: {(clips_df["laughter_prob"] > 0.1).mean():.1%}')

# Save
df.to_csv(f'{BASE}/ast_label_summary.csv', index=False)
clips_df.to_csv(f'{BASE}/ast_all_clips.csv', index=False)
print(f'\nSaved to {BASE}/')

In [ ]:
# === CREATE LABELED DATASET ===
THRESHOLD = 0.1  # Label as positive if prob > 0.1

labeled_clips = []
for r in all_results:
    for clip in r['clips']:
        labeled_clips.append({
            'video_id': r['video_id'],
            'start': clip['start'],
            'end': clip['end'],
            'laughter_prob': clip['laughter_prob'],
            'label': 1 if clip['laughter_prob'] > THRESHOLD else 0
        })

labeled_df = pd.DataFrame(labeled_clips)
print(f'Total clips: {len(labeled_df)}')
print(f'Positive clips: {labeled_df["label"].sum()} ({100*labeled_df["label"].mean():.1f}%)')

# Save
labeled_df.to_csv(f'{BASE}/ast_labeled_clips.csv', index=False)
print(f'Saved to {BASE}/ast_labeled_clips.csv')

In [ ]:
# === TRAIN CLASSIFIER ON AST LABELS ===
# Extract prosody features using FASTER method (no pyin!)

def extract_prosody_fast(waveform, sr=22050):
    """Extract prosody features WITHOUT pyin (fast!)."""
    features = []
    
    # Energy (instant) - NO pyin!
    rms = librosa.feature.rms(y=waveform)[0]
    features.extend([
        np.mean(rms), np.std(rms), 
        np.max(rms), np.min(rms),
        np.max(rms) / (np.mean(rms) + 1e-8)
    ])
    
    # ZCR (instant)
    zcr = librosa.feature.zero_crossing_rate(y=waveform)[0]
    features.extend([np.mean(zcr), np.std(zcr)])
    
    # MFCCs (fast)
    mfccs = librosa.feature.mfcc(y=waveform, sr=sr, n_mfcc=13)
    for i in range(13):
        features.extend([np.mean(mfccs[i]), np.std(mfccs[i])])
    
    # Spectral centroid (fast)
    spec_cent = librosa.feature.spectral_centroid(y=waveform, sr=sr)[0]
    features.extend([np.mean(spec_cent), np.std(spec_cent)])
    
    # Spectral bandwidth
    spec_bw = librosa.feature.spectral_bandwidth(y=waveform, sr=sr)[0]
    features.extend([np.mean(spec_bw), np.std(spec_bw)])
    
    # Spectral rolloff
    spec_rolloff = librosa.feature.spectral_rolloff(y=waveform, sr=sr)[0]
    features.extend([np.mean(spec_rolloff), np.std(spec_rolloff)])
    
    # RMS contrast (for voice/laughter distinction)
    rms_contrast = librosa.feature.spectral_contrast(y=waveform, sr=sr)
    features.extend([np.mean(rms_contrast), np.std(rms_contrast)])
    
    # Total: 5 + 2 + 26 + 2 + 2 + 2 + 2 + 2 = 43 features
    return np.array(features, dtype=np.float32)

print('Extracting prosody from labeled clips...')

# Sample for speed - take up to 5000 clips
sample_df = labeled_df.sample(n=min(5000, len(labeled_df)), random_state=42)
print(f'Processing {len(sample_df)} clips (for speed)...')

X_list = []
y_list = []
video_ids = []

for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc='Extracting'):
    video_id = row['video_id']
    start, end = row['start'], row['end']
    label = row['label']
    
    audio_path = f'{AUDIO_DIR}/{video_id}.m4a'
    if not os.path.exists(audio_path):
        continue
    
    try:
        # Load clip
        y, sr = librosa.load(audio_path, offset=start, duration=end-start, sr=22050, mono=True)
        
        # Extract fast features
        feat = extract_prosody_fast(y, sr)
        
        X_list.append(feat)
        y_list.append(label)
        video_ids.append(video_id)
        
    except Exception as e:
        continue

X = np.array(X_list)
y = np.array(y_list)
print(f'Extracted {len(X)} samples, pos={y.sum()} ({100*y.mean():.1f}%)')

In [ ]:
# === TRAIN AND EVALUATE ===
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score

# Split by video for fair evaluation
unique_videos = list(set(video_ids))
train_vids, test_vids = train_test_split(unique_videos, test_size=0.2, random_state=42)

train_mask = [v in train_vids for v in video_ids]
test_mask = [v in test_vids for v in video_ids]

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f'Train: {len(X_train)} samples, pos={y_train.sum()} ({100*y_train.mean():.1f}%)')
print(f'Test: {len(X_test)} samples, pos={y_test.sum()} ({100*y_test.mean():.1f}%)')

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train
clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_scaled, y_train)

# Evaluate
y_pred = clf.predict(X_test_scaled)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)

print(f'\n=== TEST RESULTS ===')
print(f'F1: {f1:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall: {rec:.4f}')

# Save model
import pickle
with open(f'{BASE}/ast_prosody_model.pkl', 'wb') as f:
    pickle.dump({'model': clf, 'scaler': scaler, 'features': X.shape[1]}, f)
print(f'\nModel saved to {BASE}/ast_prosody_model.pkl')
print(f'Feature dimension: {X.shape[1]}')